# 公平对比入口（默认 CPU 合成）

这是后续 Colab 实验的入口。`colab_probe.py` 和 `colab_loader_test.py` 只是环境探测。`colab_stage_e_cont.py` 会从 ep10 往下续训，不要拿来跑这四组。

默认不挂载 Drive、不下载权重或医学图像、不占用 GPU。`RUN_SYNTHETIC_TRAIN = True` 用替代小骨干跑冻结选择、梯度累积、断点恢复和一轮推理。它不是 ViT-B/16，也不能当成 GPU 证据。

`RUN_REAL = True` 才调用 `launch_real`。精度 `PRECISION = "auto"`：CPU 用 fp32；CUDA 有 bf16 用 bf16；没有 bf16 的 T4 用 fp16 加 GradScaler。点名 bf16 而卡不支持会停下。epoch 和有效 batch 不跟着改。

官方骨干是 Innerlogic-Surg `v0.1.0`（提交 `99a3e12489c11edc681cc8cc2ce4e4e481c21e4d`）里的 `surgfm_probe`。本仓库没有这份源码。没安装时探针是 BLOCKED。数据用仓库里的 `manifests_paper/train.jsonl` 和 `val.jsonl`，不要另做一份 jsonl。

本文件的自动执行只在 CPU 上跑过合成路径。没有 T4、没有官方包、没有真实权重，Colab 还不能算就绪。


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "revision").exists():
    raise SystemExit("请把工作目录设为仓库根目录后再跑")
sys.path.insert(0, str(ROOT))
MOUNT_DRIVE = False
RUN_REAL = False
RUN_SYNTHETIC_TRAIN = True
INSTALL_DEPS = False
INSTALL_BACKBONE = False
DEVICE = "cpu"  # 改成 cuda 才会申请 GPU
PRECISION = "auto"  # auto / bf16 / fp16 / fp32
SEED = 7
SEED_PHASE = "single"  # 改成 triple 才会展开三组 seed
TRAIN_MODE = "frozen_probe"  # 或 last4_finetune
BACKBONE_ID = "innerlogic_surg_vitb16"
WEIGHT_PATH = ""  # 扁平 DINOv3 预训练权重，不是 ep10 训练文件
EXPECTED_SHA = ""  # 在本地记下的 sha256，仓库不预存私有哈希
DATA_ROOT = ""
MANIFEST_DIR = str(ROOT / "manifests_paper")
OUT_DIR = "/content/ils_fair_runs" if Path("/content").exists() else "/tmp/ils_fair_runs"
print("ROOT", ROOT)
print("OUT_DIR", OUT_DIR)
print("DEVICE", DEVICE, "PRECISION", PRECISION)


In [ ]:
import subprocess
import sys
from revision.backbone_source import PIP_SPEC, probe_status
PIN = "pip install -r requirements-fair.txt --extra-index-url https://download.pytorch.org/whl/cpu"
print("工作目录", ROOT)
print("版本钉", (ROOT / "requirements-fair.txt").read_text())
print("官方骨干", PIP_SPEC)
status = probe_status()
print("backbone", status["code"], "blocked", status["blocked"], "vendored", status["vendored"])
if INSTALL_DEPS:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements-fair.txt"), "--extra-index-url", "https://download.pytorch.org/whl/cpu"])
else:
    print("Python 依赖未自动安装。需要时执行：")
    print(PIN)
if INSTALL_BACKBONE:
    subprocess.check_call([sys.executable, "-m", "pip", "install", PIP_SPEC])
    print("backbone", probe_status()["code"], probe_status().get("origin"))
else:
    print("未安装 surgfm_probe。干净环境把 INSTALL_BACKBONE 改成 True，或按 revision/backbone_source.py 的两条命令克隆 v0.1.0。")


In [ ]:
import socket
import shutil
from pathlib import Path
print("hostname", socket.gethostname())
print("同一台 VM 以这个 hostname 为准，不要凭链接判断")
probe = Path(OUT_DIR)
probe.mkdir(parents=True, exist_ok=True)
usage = shutil.disk_usage(probe)
print("path", probe, "exists", probe.exists(), "disk_free_bytes", usage.free)


In [ ]:
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Drive 已按浏览器授权挂上。请再打印一次 hostname，确认和上面是同一台。")
else:
    print("未挂载 Drive。默认不访问私人数据。")


In [ ]:
from revision.preflight import environment_report
from revision.fair_config import runtime_plan, training_matrix
report = environment_report(OUT_DIR)
print(report)
print(runtime_plan(real=RUN_REAL, cuda=report["cuda"], bf16=report["bf16"], device=DEVICE, precision=PRECISION, fp16=report["fp16"]))
print("single-seed runs", len(training_matrix(phase=SEED_PHASE, seed=SEED)))
print("选点", training_matrix()[0]["selection_score"], training_matrix()[0]["selection_protocol"])


In [ ]:
import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "revision.smoke", "--out", OUT_DIR])
print("smoke 日志", Path(OUT_DIR) / "smoke.log")


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
from revision.engine import run_synthetic_demo
if RUN_SYNTHETIC_TRAIN:
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    synth_root = Path(OUT_DIR) / f"cpu-synth-{stamp}"
    synth_root.mkdir()
    summary = run_synthetic_demo(synth_root, optimizer_steps=2)
    print("synthetic_configs", len(summary["configs"]))
    for item in summary["configs"]:
        print(item["train_mode"], "steps", item["optimizer_steps"], "backward", item["backward_passes"], "probe", item["grad_probe"])
    print("resume_match", summary["resume_match"])
    print("infer", summary["infer"]["ckpt_kind"], summary["infer"]["warnings"], summary["infer"]["out_dir"])
else:
    print("RUN_SYNTHETIC_TRAIN 为 False，跳过合成训练。")


In [ ]:
from revision.engine import launch_real
from revision.identity import RevisionError
if RUN_REAL:
    try:
        result = launch_real(
            backbone_id=BACKBONE_ID,
            weight_path=WEIGHT_PATH,
            expected_sha=EXPECTED_SHA,
            data_root=DATA_ROOT,
            manifest_dir=MANIFEST_DIR,
            out_dir=OUT_DIR,
            train_mode=TRAIN_MODE,
            device=DEVICE,
            precision=PRECISION,
            seed=SEED,
        )
        print("real_run", result["run_dir"], result["optimizer_steps"], result["autocast"], result["selection_metric"])
    except RevisionError as exc:
        print(exc.code, exc)
        raise
else:
    print("RUN_REAL 为 False。真实新训练要把它改成 True，并填 WEIGHT_PATH、EXPECTED_SHA、DATA_ROOT。MANIFEST_DIR 指向 manifests_paper。DEVICE 默认 cpu，PRECISION 默认 auto。")


## 协调方以后要跑真实实验时

1. 工作目录设为仓库根。Python 依赖：`pip install -r requirements-fair.txt --extra-index-url https://download.pytorch.org/whl/cpu`。GPU 机器改用对应 CUDA wheel。
2. 骨干不是本仓库的文件。把 `INSTALL_BACKBONE` 设为 True，或执行 `revision/backbone_source.py` 里的克隆命令，钉在 Innerlogic-Surg `v0.1.0` / `99a3e12489c11edc681cc8cc2ce4e4e481c21e4d`。装好后探针不再是 BLOCKED。不要往仓库里放一个同名假包。
3. 浏览器里打开 Colab，打印 hostname。若挂载 Drive，挂完再打印一次，两次必须相同。
4. `WEIGHT_PATH` 是扁平预训练权重，`EXPECTED_SHA` 是你在本地算的 sha256。ep10 训练文件只有 head 和微调时的 backbone，没有优化器状态，只能推理重放，不能当新训练起点。通用 DINOv3 仍要先接受许可证，不能拿 InnerLogic 文件顶替。
5. `DATA_ROOT` 指向解压后的图像根。`MANIFEST_DIR` 用仓库里的 `manifests_paper`。`train.jsonl` / `val.jsonl` 的 `split` 是难度，不是划分。测试文件不参加选点。
6. T4 没有 bf16。`PRECISION = "auto"` 会选 fp16 加 GradScaler，并写进 run config。不要把历史脚本里的 bf16 硬套到这张卡上。epoch 仍是 6，有效 batch 仍是 8。
7. 替代小骨干和这份 CPU 执行都不是真实 ViT 或 T4 证据。官方包、权重、数据和 GPU 齐了之前，Colab 还没有就绪。
